# Lab 4 — Multi-Agent Orchestration (🟡 Builder rail)

> 🟡 **Builder rail.** Run the cells top to bottom. The lines marked `# 👉` are the parts you wire up; the `# TODO (bonus)` cell is optional. Same checkpoint as the other rails.

> 📄 **Lab instructions:** [../labs/lab-04.md](../labs/lab-04.md) (all three rails) · 🟢 portal walkthrough: [../labs/lab-04-portal.md](../labs/lab-04-portal.md)

**Goal:** turn the single triage agent from Labs 1–3 into an **orchestrator** that, for a LOW‑risk *compound* request, **delegates** to two specialist agents (Education + Follow‑Up) and **synthesises one combined reply**.

> ⚠️ **Prerequisite:** the HealthHub docs must be in `../knowledge/healthhub-discharge-pack/` — the Education specialist grounds its answer in them, exactly like Lab 2. Run from the `assets/` folder so `common` imports resolve: `python lab4_multiagent.py`.

### What's new versus Labs 1–3? (the tech stack, in plain terms)

So far Care Pal was **one** agent doing everything. Real assistants split work across **specialists** plus a **coordinator** — the same pattern as a triage nurse who refers you to the right department. Here the triage agent *becomes* that coordinator.

| Building block | What it is | How it interacts with Microsoft Foundry |
|----------------|-----------|-----------------------------------------|
| **Orchestrator agent** | The triage agent + extra rules telling it *when* to delegate and how to merge results into one reply. | A normal `PromptAgentDefinition` in the shared Foundry project — the same object type as every earlier lab, just with more instructions and tools. |
| **Specialist agents** | Two smaller agents: **Education** (grounded with file search, like Lab 2) and **Follow‑Up** (plain scheduling help). | Each is its own agent version created via `project.agents.create_version(...)` — independently versioned and deletable in Foundry. |
| **Function tools** (`FunctionTool`) | `ask_education` and `ask_followup` — tool *declarations* the orchestrator can call, each with a tiny JSON schema. | Attached to the orchestrator's definition in `tools=[...]`. Foundry advertises these tools to the model and returns a **function‑call request** when it wants one. |
| **Function‑tool loop** (`run_with_trace`) | The client‑side loop that receives each function‑call request, runs the matching specialist, and feeds the result back until the orchestrator produces its final JSON. | This is *client‑side* orchestration: Foundry emits the calls; your code executes them and continues the same response with `previous_response_id`. |
| **Delegation trace** | A record of which tools fired, so we can *assert* the orchestrator really delegated (≥ 2 specialist calls). | Read from the response's `function_call` items — the code‑first equivalent of watching tool calls in the portal's thread view. |

> 🔀 **Why function tools and not `ConnectedAgentTool`?** In the *new* Foundry agents API the classic `ConnectedAgentTool` is retired. The portal‑aligned ways to build multi‑agent systems are now: **(1) delegate via function tools** (this lab — runnable and assertable), **(2) Workflow agents** (`WorkflowAgentDefinition`) for declarative graphs, and **(3) the A2A tool** (`A2APreviewTool`) for cross‑project / external agents. See the **GO FURTHER** note at the bottom of the last cell.

> 📚 **Where these patterns come from (optional reading):** the `azure-ai-projects` 2.x samples `tools/sample_agent_function_tool.py` and `sample_workflow_multi_agent.py`, plus the *agent‑framework* workflows (a code‑first option). The `common/` helpers already apply them for you.

## Cell 1 — Make the shared helpers importable

Same setup step as the earlier labs: add this notebook's folder (`assets/`) to Python's import path so the next cell can do `from common.carepal_common import ...`. It works whether you run this as a notebook or as the `python lab4_multiagent.py` script. Nothing touches Foundry yet — pure local setup.

In [1]:
# --- make `common` importable whether run as a script or a notebook ---
import sys
import pathlib
_here = (pathlib.Path(globals()["__file__"]).resolve().parent
         if "__file__" in globals() else pathlib.Path.cwd())
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

## Cell 2 — Import the helpers and write the orchestrator + specialist instructions

This cell brings in the helpers and defines the **three roles** in our multi‑agent system — the orchestrator and its two specialists.

| Helper | What it does with Foundry |
|--------|---------------------------|
| `make_triage_agent` | Creates a triage‑style agent with the structured 7‑key JSON contract — used for both the **orchestrator** and the **Education** specialist. |
| `create_agent` | Creates a plain agent (no JSON schema) — used for the **Follow‑Up** specialist. |
| `build_vector_store` / `file_search_tool` | Rebuild the Lab 2 knowledge index and wrap it as a `FileSearchTool` so the Education specialist can **cite HealthHub**. |
| `function_tool` | Declares a `FunctionTool` (`ask_education`, `ask_followup`) the orchestrator can call to reach a specialist. |
| `run_with_trace` / `run_text` | `run_text` runs a single specialist; `run_with_trace` runs the orchestrator through the **function‑tool loop** and records which tools fired. |
| `text_of` / `agent_name` / `cleanup` | Look up the test message; build a unique per‑attendee agent name; delete every agent afterwards. |
| `TRIAGE_INSTRUCTIONS` | The Lab 1 triage rules the orchestrator builds on top of. |

- **`ORCHESTRATOR`** extends the triage rules with a *delegation policy*: triage first, and for LOW‑risk paths **call `ask_education` for self‑care content and `ask_followup` for scheduling / check‑ins**, then merge any `source_labels` / `source_urls` the specialists return into one final JSON. For medium/high risk it must **not** delegate — safety escalation stays in the orchestrator itself.
- **`FOLLOWUP`** is the plain instruction for the Follow‑Up specialist (schedules check‑ins, suggests appointment types, collects symptom responses, never diagnoses).
- **`_QUESTION_SCHEMA`** is the tiny JSON contract both tools share — a single `question` string the orchestrator forwards to the specialist.

In [2]:
from common.carepal_common import (
    create_agent,
    make_triage_agent,
    run_with_trace,
    run_text,
    build_vector_store,
    file_search_tool,
    function_tool,
    text_of,
    agent_name,
    cleanup,
    TRIAGE_INSTRUCTIONS,
)

ORCHESTRATOR = TRIAGE_INSTRUCTIONS + """
You are Care Pal's orchestrator. Triage first. For LOW-risk paths, delegate then synthesise ONE reply:
- education / self-care content -> call the `ask_education` tool
- follow-up scheduling, check-ins, symptom tracking -> call the `ask_followup` tool
Call only the specialists needed. For medium/high risk do NOT delegate. Merge any source_labels /
source_urls returned by specialists into your final JSON.
"""

FOLLOWUP = (
    "You schedule check-ins and suggest follow-up appointment types after discharge, and collect "
    "symptom responses. You do not diagnose. Reply briefly and plainly."
)

# A tiny JSON schema shared by both delegation tools.
_QUESTION_SCHEMA = {
    "type": "object",
    "properties": {"question": {"type": "string", "description": "the user's request to forward"}},
    "required": ["question"],
    "additionalProperties": False,
}

## Cell 3 — Build the specialists, wire the tools, and prove delegation

This is the heart of the lab — the full multi‑agent round‑trip with Microsoft Foundry:

1. **Build the knowledge index** — `build_vector_store(...)` + `file_search_tool(...)` recreate the Lab 2 vector store so the Education specialist can ground its answer in HealthHub.
2. **Create the two specialists** — `education` (triage‑structured **+** file search) and `followup` (plain) are each created as their own agent version in the shared project.
3. **Expose them as function tools** — `ask_education` and `ask_followup` become `FunctionTool`s on the orchestrator. The `functions` dict maps each tool name to a Python handler that forwards the question to the matching specialist via `run_text` — here wrapped in a small logger so you can **watch each hand‑off** (the question forwarded and the specialist's reply) instead of only seeing the tool names.
4. **Create the orchestrator** — `make_triage_agent(instructions=ORCHESTRATOR, tools=tools, structured=True)` provisions it with the delegation rules **and** both tools.
5. **Run the compound request** — `run_with_trace(orchestrator, text_of("follow_up_and_diet"), functions=...)` sends *"What follow‑up appointments does my father need after heart failure, and what diet should he keep?"*. Foundry decides it needs both specialists, emits two `function_call`s, and your loop runs them and feeds the results back until the orchestrator returns one merged JSON.
6. **Assert delegation** — the captured trace must show **≥ 2 specialist calls** (`ask_education` **and** `ask_followup`), proving the orchestrator really split the work. Then it prints the merged reply and `Lab 4 passed ✅`.

All three agents are **left running on purpose** so you can inspect them in the portal (Cell 4); you delete them in the final cleanup cell.

**Expected output:** the run now **logs every hand‑off** so you can see the delegation as it happens — the slice of the question sent to each specialist and that specialist's reply — then the tool‑call list and the orchestrator's **final merged JSON** (one answer combining diet guidance **and** the follow‑up plan):
```
Created orchestrator: carepal-<initials>-orchestrator (version 1)

─── ask_education  ▸ orchestrator asked ───
...the education slice of the question...
─── ask_education  ◂ specialist replied ───
{ ...grounded self-care JSON with healthhub.sg citations... }

─── ask_followup  ▸ orchestrator asked ───
...the scheduling slice of the question...
─── ask_followup  ◂ specialist replied ───
...a plain follow-up plan...

tool calls: ['ask_education', 'ask_followup']

=== Orchestrator's final merged reply (synthesised from both specialists) ===
{ ...one 7-key triage JSON combining both... }

Lab 4 passed ✅
```

> 🔍 **Want the raw request/response too?** `run_with_trace` also returns the underlying `trace.response` object (the OpenAI‑style response with the tool‑call items). Inspect `handoffs` for the structured `{tool, question, reply}` records, or `trace.response.output` for the raw function‑call payloads Foundry emitted.

> ⚙️ **If your model rejects structured output + tools together:** set `structured=False` on the orchestrator (see the inline `NOTE`). `extract_json()` still recovers the JSON from a prose‑wrapped reply.

> 💡 The `# TODO (bonus)` note is optional: add a **third** specialist (e.g. Assessment or Enrollment & Linkage) and show it firing in the trace.

> 🚀 **GO FURTHER (Engineer):** re‑implement this as a **Workflow agent** (`WorkflowAgentDefinition`, see `sample_workflow_multi_agent.py`) or with the **Microsoft Agent Framework** (`FoundryChatClient` + workflows) for declarative, code‑first orchestration.

### 🧭 Three ways to orchestrate (and when to pick each)

This lab uses **function tools**, but it's one of three portal‑aligned patterns. They differ in *who decides the route* and *where the flow runs* — that's the real design choice.

| Pattern | Who decides the route | Where the flow runs | Reach for it when… |
|---------|-----------------------|---------------------|--------------------|
| **Function tools** *(this lab)* | The **model**, at runtime — the orchestrator calls only the specialists it needs. | **Client‑side** loop (`run_with_trace`) in your process. | You want **dynamic, model‑driven** delegation that's easy to run and *assert* in a notebook. |
| **Workflow agent** (`WorkflowAgentDefinition`) | **You**, declaratively — you draw the graph (e.g. the portal's *Sequential* template: triage → education → follow‑up). | **Foundry** (hosted) — it runs and traces the graph for you. | The flow is **structured and repeatable** and you want portal visibility, versioning, and governance. |
| **Microsoft Agent Framework** (`FoundryChatClient` + workflows) | **You**, imperatively in code — arbitrary loops, branching, custom state. | **Your app** — Foundry supplies the models/agents; your code owns the control flow. | Orchestration is **complex/dynamic** and belongs inside a larger codebase. |

> 🔎 **Contrast with the portal (Navigator) lab:** that lab builds a **Sequential Workflow**, which runs *every* node on *every* turn — perfect for *showing* multi‑agent. The function‑tool orchestrator here instead lets the model **choose when to delegate** (and skip it entirely for medium/high‑risk cases). None is "more correct" — it's a trade‑off between model‑driven flexibility, declarative auditability, and code‑first control.

In [4]:
import json

# 👉 Build the Lab 2 knowledge index and create the two specialists.
vs_id = build_vector_store("healthhub-discharge-pack")
fs = file_search_tool(vs_id)

# Two specialists: Education (grounded with file search) + Follow-Up (plain).
education = make_triage_agent(
    name=agent_name("education"),
    instructions=TRIAGE_INSTRUCTIONS,
    tools=[fs],
    structured=True,
)
followup = create_agent(name=agent_name("followup"), instructions=FOLLOWUP, structured=False)

# 👉 Expose each specialist as a FUNCTION TOOL the orchestrator can call.
tools = [
    function_tool("ask_education", "Grounded self-care / education answer with citations",
                  _QUESTION_SCHEMA),
    function_tool("ask_followup", "Follow-up scheduling, check-ins and symptom tracking",
                  _QUESTION_SCHEMA),
]

# Handlers: each forwards the question to its specialist and returns the text.
# We wrap them in a small logger so you can SEE the hand-off (the question the
# orchestrator forwarded and the specialist's reply), not just the tool names.
# `handoffs` also keeps a structured record you can inspect afterwards.
handoffs = []


def _delegate(tool_name, specialist):
    def handler(question):
        reply = run_text(specialist, question)
        handoffs.append({"tool": tool_name, "question": question, "reply": reply})
        print(f"\n─── {tool_name}  ▸ orchestrator asked ───\n{question}")
        print(f"─── {tool_name}  ◂ specialist replied ───\n{reply}")
        return reply
    return handler


functions = {
    "ask_education": _delegate("ask_education", education),
    "ask_followup": _delegate("ask_followup", followup),
}

# 👉 Create the orchestrator. It's kept as a top-level variable (like Labs 1-2) so you
# can inspect it in the Foundry portal and then delete it from the separate cleanup cell.
# NOTE: if your model rejects structured output + tools together, set structured=False here;
# extract_json() still tolerates a JSON-in-prose reply.
orchestrator = make_triage_agent(
    name=agent_name("orchestrator"), instructions=ORCHESTRATOR, tools=tools, structured=True
)
print(f"Created orchestrator: {orchestrator.name} (version {orchestrator.version})")

out, trace = run_with_trace(orchestrator, text_of("follow_up_and_diet"), functions=functions)
called = [c.name for c in trace.tool_calls]
specialist = [c for c in trace.tool_calls if c.name in ("ask_education", "ask_followup")]
print("\ntool calls:", called)
assert len(specialist) >= 2, f"expected >=2 specialist calls, got {called}"

# The orchestrator MERGES both specialist replies into one triage JSON - print it so you
# can see the synthesised, single answer the user would actually receive.
print("\n=== Orchestrator's final merged reply (synthesised from both specialists) ===")
print(json.dumps(out, indent=2))
print("\nLab 4 passed ✅")

# TODO (bonus): add a third specialist (Assessment or Enrollment & Linkage) and show it firing.
# GO FURTHER (Engineer): re-implement this as a Workflow agent (WorkflowAgentDefinition,
# see sample_workflow_multi_agent.py) or with the Microsoft Agent Framework
# (agent-framework -> FoundryChatClient + workflows) for code-first orchestration.

Created orchestrator: carepal-jb-orchestrator (version 1)

─── ask_followup  ▸ orchestrator asked ───
What follow-up appointments does a discharged heart failure patient typically need in Singapore? Please include recommended timing (e.g., within 1-2 weeks), which specialists (cardiology, heart failure clinic, primary care), tests (blood tests, renal function, weights), and frequency for ongoing reviews.
─── ask_followup  ◂ specialist replied ───
In Singapore, patients discharged after a heart failure admission are usually scheduled for several early and ongoing follow‑ups to reduce readmission risk and adjust medications safely.

Early follow‑up (first 1–2 weeks after discharge)
- Heart failure clinic or cardiology review: usually within 7–14 days. Many public hospitals (e.g., NHCS, NUH, TTSH, SGH) have dedicated HF nurse clinics.
- What is checked: symptoms, blood pressure, heart rate, daily weight trend, medication tolerance.
- Tests often ordered: blood tests for renal function (cr

### 📤 Reading the output — where the hand-off actually happens

The log above **is** the multi-agent hand-off, captured step by step. Reading it top to bottom:

1. **`Created orchestrator: ...`** — the coordinator agent now exists in Foundry alongside the two specialists. On its own, nothing has been delegated yet.
2. **`─── ask_followup ▸ orchestrator asked ───`** — this is the **first hand-off**. Notice the orchestrator didn't forward your raw question verbatim; it *reformulated* a focused scheduling sub-question (*"What follow-up appointments… timing… specialists… tests…?"*) and routed it to the Follow-Up specialist. That specialist replied with a plain-language appointment/test plan.
3. **`─── ask_education ▸ orchestrator asked ───`** — the **second hand-off**. The orchestrator split off the *diet* half of the question and sent it to the Education specialist, which replied with structured triage JSON (sodium/fluid/potassium guidance, Singapore tips).
4. **`tool calls: ['ask_followup', 'ask_education']`** — the delegation trace. Two *different* specialists fired, which is exactly what the `assert len(specialist) >= 2` line checks. **This list is the proof** the orchestrator decomposed one compound question and delegated the pieces — if it had answered alone, this would be empty.
5. **`=== Orchestrator's final merged reply ===`** — the payoff. The orchestrator **synthesised both specialist replies into one 7-key triage JSON**: the `reply` field weaves the follow-up schedule *and* the diet advice into a single answer, adds the safety escalation line, and keeps the triage contract (`intent`, `risk_level`, `route`, …). This one object is what the user would actually receive.

**Why this demonstrates multi-agent orchestration:** three agents collaborated on a single turn — the orchestrator *decided* the request needed two specialists, *reformulated* a targeted sub-question for each, ran them (steps 2–3), and *merged* their outputs into one coherent reply (step 5). The `tool calls` line is your machine-checkable receipt that the delegation really occurred, rather than one model faking a combined answer.

> 💡 Want to see the exact text passed between agents? The `handoffs` list now holds `[{tool, question, reply}, …]` for each delegation — e.g. run `handoffs[0]["question"]` to see the sub-question sent to the first specialist.

## Cell 4 — Inspect your multi-agent system in the Foundry portal (optional pause)

Your three agents — **`carepal-<initials>-orchestrator`**, **`-education`**, and **`-followup`** — are now **live in the shared Foundry project**. The cleanup step is intentionally in the *next* cell so you can go look at them first.

1. Open **https://ai.azure.com** → project **`research-workshop`** → **Agents**.
2. Find your three agents (filtered by your initials). Open the **orchestrator** and notice its **tools** list — the `ask_education` and `ask_followup` function tools you attached from code.
3. Open the **education** specialist and check its **Knowledge / Tools** section — it has the same file‑search vector store you built in Lab 2.
4. Try the compound question in the orchestrator's **Playground** and watch it fan out to both specialists, then merge one reply — the same delegation the trace asserted above.

> ℹ️ **No conflict if you re-run:** re-running the cell above creates a **new version** of each agent (Foundry versions agents by name). Your initials make the names unique in the shared project, so you never clash with other participants.

In [5]:
# 🧹 Run this LAST — once you've finished inspecting the agents in the portal.
# Deletes all three agent versions so the shared project stays tidy. Safe to run
# even if you re-created them a few times; it removes the current versions.
cleanup(orchestrator, education, followup)
print("Cleaned up orchestrator + specialists ✅")

Cleaned up orchestrator + specialists ✅
